# RNG Cost calculation notebook 

Start date: 09/08/2026
Author: Sahar H. El Abbadi

## Setup 

Import packages, load global variables and datasets

In [ ]:
# Import packages 
import pandas as pd 
import pathlib

# Load variables 

# Physical constants
MMBTU_TO_MJ = 1055.06 # 1mmBtu = 1055.06MJ
HHV1 = 55.6 # MJ/KgCH4 (El Abbadi, Preprint)
P = 1 # Pressure(P) atm
T = 298 # Temperature(T) K
R = 0.0821 # Ideal gas constant(R) L.atm/mol.K

# Methane lower heating value - when water is not condensed 
KJ_PER_MOL_CH4 = 890.3   # kJ/mol, reported in Harrison et al., 2010
MW_CH4 = 16.04    # g/mol = 0.01604 kg/mol

# Convert kJ/mol to MJ/kg
METHANE_MJ_PER_KG = (KJ_PER_MOL_CH4 / 1000) / (MW_CH4 / 1000) # Result: 55.6 MJ / kg



In [3]:
# Load data 

# Facilities with AD in the United States 
ad_data = pathlib.Path("biogas-upgrading-project", 
                       "clean_data",
                       "ad_data.csv")

## Helper functions 

In [ ]:
## Standard units for material flows:
#  
# Flow rate: m3/day unless otherwise explicitly specified 
# Biogas production: kgCH4 / hr unless otherwise explicitly specified 
# RNG production: kgCH4 / hr unless otherwise explicitly specified 

#### Unit conversion equations 

## 1) Convert mmBTU of methane to kgCH4

def convert_mmBTU_to_kgCH4(mmBTU): 
    """Converts methane from units of mmBTU to kg CH4

    Parameters
    -----------
    mmBTU: float
        Natural gas / RNG energy content 

    Returns 
    -------
    float
        Calculated methane as kgCH4

    """

    # Convert from mmBTU to MJ 
    mj = mmBTU * MMBTU_TO_MJ

    # Convert from MJ to kg CH4
    kgCH4 = mj * (1/METHANE_MJ_PER_KG)

    return kgCH4

## 2) Convert kgCH4 to mmBTU

def convert_kgCH4_to_mmBTU(kgCH4): 
    """Converts methane from units of mmBTU to kg CH4

    Parameters
    -----------
    kgCH4: float
        Natural gas / biogas as kg CH4 

    Returns 
    -------
    float
        Calculated energy content of methane as mmBTU

    """
    # Convert from kg CH4 to MJ
    mj = kgCH4 * METHANE_MJ_PER_KG

    # Convert from MJ to mmBTU 
    mmBTU = mj * (1 / MMBTU_TO_MJ)

    return mmBTU



#### Mass flow equations

## 1) Calculate biogas production from facility flow rate 

def calc_biogas_from_ww_ad(flow_rate): # Calculates biogas flowrate in KgCH4/h
    """ Calculates biogas production rate for a wastewater treatment plant based on facility flow rate. 
    Source: El Abbadi et al, 2026; empirically derived 

    Parameters
    -----------
    flow_rate : float 
        Wastewater treatment plant flow rate in m3 / day

    Returns 
    -------
    float 
        Calculated biogas production rate in kgCH4 / hour
    """

    return flow_rate*0.00148




## 2) Calculate RNG production from biogas production 

def calc_rng_from_upgrading(biogas_input, efficiency=0.9): # Calculates RNG production in kgCH4/hr
    """ Calculate RNG produced from upgrading unit based on input efficiency 

    Parameters
    -----------
    biogas_input : float 
        Biogas production rate in kg CH4 / hour 
    
    efficiency : float 
        Upgrading efficiency, default value of 90%

    Returns 
    -------
    float 
        Calculated RNG production rate in kgCH4 / hour
    """

    return biogas_input * efficiency


## Calculate RNG production

For a given flow rate of a wastewater treatment plant (m3 / day), calculate expected RNG production 

In [ ]:
# Insert code here that uses the functions above to calculate biogas production from flow rate

## Calculate RNG upgrading cost 

In [ ]:
# Cost estimate using Parker et al., 2017

# Parker et al Table 1 reports: 
## Upgrading to pipeline 
## Capital cost = 1_370_000 * RNG_Flow^0.56
## Annual O&M = 101_625 * RNG_Flow^0.81

# RNG refers to RNG produced and is in units of mmBtu / hr 

# Function for Parker et al 

def calc_capex_parker(biogas_produced_kgCH4): 
    """ Calculate capital cost of biogas upgrading to RNG for a given biogas production rate 

    Parameters
    -----------
    biogas_produced_kgCH4 : float 
        RNG production rate in kg CH4 / hour 

    Returns 
    -------
    float 
        Calculated capital cost in USD 
    """

    # 1) Convert RNG in kgCH4/hr to mmBTU per hr 
    biogas_produced_mmBTU = convert_kgCH4_to_mmBTU(biogas_produced_kgCH4)

    # 2) Convert biogas produced (MMBtu) to RNG produced. Use default efficiency value for now 
    rng_produced_mmBTU = calc_rng_from_upgrading(biogas_produced_mmBTU)

    # 3) Apply Parker et al equation 

    upgrading_capex = 1_370_000 * rng_produced_mmBTU^0.56

    return upgrading_capex